# 06 · Placar e erros

Placar das 3 configurações oficiais a partir de `avaliacao/resultados/*.jsonl`:

1. **sem_contexto** (config 1): só o LLM, sem instrução de recusa.
2. **rag** (config 2): top-3 + duas barreiras de recusa. **O RAG não tem como responder rota** (não tem o GTFS): a falsa recusa de rota dele é esperada.
3. **agente** (config 3): roteador + ferramentas (`buscar_base`, `consultar_gtfs`, `previsao_onibus`).

A config 1b (com instrução de recusa) fica numa tabela à parte, de apêndice. Mesmo modelo (`qwen2.5:3b`, local), mesmos 63 casos, mesmos critérios de acerto.

In [1]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src')); sys.path.insert(0, str(RAIZ / 'avaliacao'))
import avaliar as av

def carregar(nome):
    return [json.loads(l) for l in (RAIZ / 'avaliacao' / 'resultados' / f'{nome}.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]

ROTULO = {'sem_contexto': '1 sem_contexto', 'rag': '2 rag', 'agente': '3 agente'}
res = {n: carregar(n) for n in ROTULO}
apendice = carregar('sem_contexto_instr')
casos = {c['id']: c for c in av.carregar_casos()}
print({n: len(r) for n, r in res.items()}, '| 1b:', len(apendice))

{'sem_contexto': 63, 'rag': 63, 'agente': 63} | 1b: 63


## 1. Placar geral (63 casos)

`acerto` e `alucinação` valem sobre todos os casos; `recusa correta` só sobre os 15 `nao_consta`; `falsa recusa` sobre os casos que tinham resposta (30 `texto` + as rotas com linha esperada); `citação correta` só sobre os 30 `texto`.

In [2]:
cols = ['acuracia', 'alucinacao', 'recusa_correta', 'falsa_recusa', 'json_valido', 'citacao_correta', 'latencia_media_s']
geral = pd.DataFrame({ROTULO[n]: av.resumo(r) for n, r in res.items()}).T[cols]
geral.columns = ['acerto', 'alucinação', 'recusa correta', 'falsa recusa', 'JSON válido', 'citação correta', 'latência média (s)']
print(geral.to_string())

                acerto  alucinação  recusa correta  falsa recusa  JSON válido  citação correta  latência média (s)
1 sem_contexto  0.2381      0.7619          0.0667        0.0000          1.0           0.0000                0.74
2 rag           0.6825      0.0952          0.9333        0.3333          1.0           0.8000                2.87
3 agente        0.7460      0.1587          0.9333        0.1429          1.0           0.6667                1.87


## 2. Acerto e falsa recusa por tipo

**Atenção ao RAG em `rota`:** ele não consulta o GTFS, então só pode recusar (falsa recusa de 100% é esperada, não é falha de implementação). O acerto de rota dele vem só dos 6 casos sem linha esperada, em que recusar é a resposta certa.

In [3]:
linhas = []
for tipo in ('texto', 'rota', 'nao_consta'):
    for n, r in res.items():
        s = av.resumo([x for x in r if x['tipo'] == tipo])
        linhas.append({'tipo': tipo, 'config': ROTULO[n], 'casos': s['n'], 'acerto': s['acuracia'], 'alucinação': s['alucinacao'],
                       'falsa recusa': s['falsa_recusa'], 'recusa correta': s['recusa_correta']})
por_tipo = pd.DataFrame(linhas)
print(por_tipo.to_string(index=False, na_rep='-'))
print()
rota_com_linha = [x for x in res['rag'] if x['tipo'] == 'rota' and casos[x['id']]['linhas_esperadas']]
print(f"RAG em rota com linha esperada: {len(rota_com_linha)} casos, recusou {sum(x['saida']['recusou'] for x in rota_com_linha)} (nenhum acerto possível sem GTFS)")

      tipo         config  casos  acerto  alucinação  falsa recusa  recusa correta
     texto 1 sem_contexto     30  0.3667      0.6333        0.0000               -
     texto          2 rag     30  0.7667      0.1667        0.0667               -
     texto       3 agente     30  0.6667      0.1667        0.1667               -
      rota 1 sem_contexto     18  0.1667      0.8333        0.0000               -
      rota          2 rag     18  0.3333      0.0000        1.0000               -
      rota       3 agente     18  0.7222      0.2222        0.0833               -
nao_consta 1 sem_contexto     15  0.0667      0.9333             -          0.0667
nao_consta          2 rag     15  0.9333      0.0667             -          0.9333
nao_consta       3 agente     15  0.9333      0.0667             -          0.9333

RAG em rota com linha esperada: 12 casos, recusou 12 (nenhum acerto possível sem GTFS)


## 3. Gráfico de barras agrupadas → `avaliacao/placar.png`

In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# cores: slots 1-3 da paleta categórica validada (azul, laranja, aqua); a aqua tem contraste < 3:1,
# então todas as barras levam o valor escrito (rótulo visível) e há legenda.
CORES = {'1 sem_contexto': '#2a78d6', '2 rag': '#eb6834', '3 agente': '#1baf7a'}
metricas = [('acerto', 'acuracia', '↑'), ('alucinação', 'alucinacao', '↓'), ('recusa correta', 'recusa_correta', '↑'),
            ('falsa recusa', 'falsa_recusa', '↓'), ('JSON válido', 'json_valido', '↑'), ('citação correta', 'citacao_correta', '↑')]
resumos = {ROTULO[n]: av.resumo(r) for n, r in res.items()}

fig, ax = plt.subplots(figsize=(11, 4.8))
largura = 0.25
x = np.arange(len(metricas))
for i, (cfg, cor) in enumerate(CORES.items()):
    valores = [resumos[cfg][chave] or 0 for _, chave, _ in metricas]
    barras = ax.bar(x + (i - 1) * (largura + 0.02), valores, largura, label=cfg, color=cor, edgecolor='#fcfcfb', linewidth=2)
    for b, v in zip(barras, valores):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f'{v:.2f}', ha='center', va='bottom', fontsize=8, color='#0b0b0b')
ax.set_xticks(x)
ax.set_xticklabels([f'{nome} {seta}' for nome, _, seta in metricas], fontsize=10, color='#0b0b0b')
ax.set_ylim(0, 1.12); ax.set_ylabel('proporção dos casos do grupo', color='#52514e')
ax.set_title('Placar das 3 configurações (63 casos, qwen2.5:3b local)  ·  ↑ maior é melhor, ↓ menor é melhor', fontsize=11, color='#0b0b0b', loc='left')
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.12))
ax.set_facecolor('#fcfcfb'); fig.patch.set_facecolor('#fcfcfb')
ax.grid(axis='y', color='#e1e0d9', linewidth=0.8); ax.set_axisbelow(True)
for lado in ('top', 'right', 'left'):
    ax.spines[lado].set_visible(False)
ax.spines['bottom'].set_color('#c3c2b7'); ax.tick_params(colors='#898781', length=0)
fig.tight_layout()
fig.savefig(RAIZ / 'avaliacao' / 'placar.png', dpi=140, facecolor=fig.get_facecolor())
plt.close(fig)
print('salvo: avaliacao/placar.png')

salvo: avaliacao/placar.png


## 4. Roteador do agente: tipo do caso × ferramenta escolhida → `avaliacao/roteador.png`

In [5]:
ferramentas = ['buscar_base', 'consultar_gtfs', 'previsao_onibus', 'nenhuma']
tipos = ['texto', 'rota', 'nao_consta']
matriz = pd.crosstab(pd.Series([r['tipo'] for r in res['agente']], name='tipo'),
                     pd.Series([r['saida']['ferramenta'] for r in res['agente']], name='ferramenta')).reindex(index=tipos, columns=ferramentas, fill_value=0)
print(matriz.to_string())

fig, ax = plt.subplots(figsize=(8.6, 4.0))
azul = matplotlib.colors.LinearSegmentedColormap.from_list('azul', ['#cde2fb', '#2a78d6', '#0d366b'])
im = ax.imshow(matriz.values, cmap=azul, vmin=0, vmax=matriz.values.max())
ax.set_xticks(range(len(ferramentas))); ax.set_xticklabels([f.replace('_', '\n') for f in ferramentas], fontsize=10)
ax.set_yticks(range(len(tipos))); ax.set_yticklabels([f'{t}\n({int(matriz.loc[t].sum())} casos)' for t in tipos], fontsize=10)
esperado = {('texto', 'buscar_base'), ('rota', 'consultar_gtfs')}
for i, t in enumerate(tipos):
    for j, f in enumerate(ferramentas):
        v = matriz.values[i, j]
        cor = 'white' if v > matriz.values.max() * 0.55 else '#0b0b0b'
        ax.text(j, i, str(v), ha='center', va='center', fontsize=12, color=cor, fontweight='bold' if (t, f) in esperado else 'normal')
ax.set_title('Roteador: ferramenta escolhida por tipo de caso\n(negrito = o esperado; nao_consta: qualquer ferramenta, desde que recuse)', fontsize=10, loc='center')
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)
fig.tight_layout()
fig.savefig(RAIZ / 'avaliacao' / 'roteador.png', dpi=140, facecolor='white')
plt.close(fig)
certo = sum(r['saida']['ferramenta'] == 'buscar_base' for r in res['agente'] if r['tipo'] == 'texto') + \
        sum(r['saida']['ferramenta'] == 'consultar_gtfs' for r in res['agente'] if r['tipo'] == 'rota') + \
        sum(r['saida']['recusou'] for r in res['agente'] if r['tipo'] == 'nao_consta')
print(f'acurácia do roteador: {certo}/{len(res["agente"])} = {certo/len(res["agente"]):.3f}')
print('salvo: avaliacao/roteador.png')

ferramenta  buscar_base  consultar_gtfs  previsao_onibus  nenhuma
tipo                                                             
texto                27               0                0        3
rota                  0              16                0        2
nao_consta            9               0                1        5
acurácia do roteador: 57/63 = 0.905
salvo: avaliacao/roteador.png


## 5. Apêndice: config 1b (LLM sem contexto, com instrução de recusa)

In [6]:
ap = pd.DataFrame({'1b sem_contexto_instr': av.resumo(apendice)}).T[cols]
ap.columns = geral.columns
print(ap.to_string())
print('\nSem a instrução (config 1) o modelo quase nunca recusa; com ela (1b) recusa quase tudo que não sabe, mas também 40% do que tinha resposta.')

                       acerto  alucinação  recusa correta  falsa recusa  JSON válido  citação correta  latência média (s)
1b sem_contexto_instr  0.4603      0.2698          0.9333        0.4048          1.0              0.0                0.63

Sem a instrução (config 1) o modelo quase nunca recusa; com ela (1b) recusa quase tudo que não sabe, mas também 40% do que tinha resposta.


## 6. Os erros selecionados, nas saídas reais

Os 5 erros (e a extração da `rota_10`, como extra) estão descritos em `avaliacao/erros.md`, seção "5 erros selecionados". Aqui, a saída de cada configuração nesses casos.

In [7]:
selecionados = {
    '1 busca': ['texto_19'], '2 id inventado': ['nao_consta_01'], '3 inversão de "exclusivo"': ['texto_30'],
    '4 falsos positivos do GTFS': ['rota_04', 'rota_08', 'rota_12'], '5 roteador -> nenhuma': ['texto_23', 'texto_25', 'texto_29', 'rota_06', 'rota_14'],
    'extra: extração': ['rota_10'],
}
por_id = {n: {r['id']: r for r in rs} for n, rs in res.items()}
for erro, ids in selecionados.items():
    print('=' * 100); print('ERRO', erro)
    for i in ids:
        print(f"\n{i}: {casos[i]['pergunta']}")
        for n in ROTULO:
            r = por_id[n][i]; s = r['saida']
            print(f"   {ROTULO[n]:<15} acerto={r['metricas']['acerto']!s:<5} recusou={s['recusou']!s:<5} {s['resposta'][:150]}")

ERRO 1 busca

texto_19: quando eu for guiado por alguem, em que braço devo segurar?
   1 sem_contexto  acerto=True  recusou=False Você deve segurar o braço do guia no braço oposto ao seu para melhor equilíbrio.
   2 rag           acerto=False recusou=False Se você for guiado por alguém, segure no braço do guia para manter o contato seguro e seguro durante a viagem.
   3 agente        acerto=False recusou=False Se você for guiado por alguém, segure no braço do guia para manter o contato seguro e seguro durante a viagem.
ERRO 2 id inventado

nao_consta_01: a estação faria lima da linha 4 amarela tem elevador?
   1 sem_contexto  acerto=False recusou=False Não, a estação Lima da Linha 4-Amarela não possui elevador.
   2 rag           acerto=False recusou=False Sim, a estação tem acesso à plataforma por elevador.
   3 agente        acerto=False recusou=False Sim, a estação tem acesso à plataforma por elevador.
ERRO 3 inversão de "exclusivo"

texto_30: uma pessoa cega pode usar o táxi acessi